# Lesson 1 — Density Matrices and Mixed States

**Quantum Computing with Qiskit II**

This notebook accompanies Lesson 1. Work through the cells in order.

**Topics covered:**
- Pure states as density matrices and the role of coherences
- Mixed states, ensembles, and purity
- Partial trace and entanglement entropy
- The `DensityMatrix` class in Qiskit
- Fidelity and trace distance

In [ ]:
!pip install qiskit qiskit-aer --quiet

In [ ]:
import qiskit
import qiskit_aer

print("Qiskit:    ", qiskit.__version__)
print("Qiskit Aer:", qiskit_aer.__version__)

In [ ]:
# Core imports used throughout the notebook
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.quantum_info import (
    DensityMatrix, Statevector,
    partial_trace, entropy, state_fidelity
)

print("Imports complete.")

## 1. Pure vs Mixed States

A **pure state** is a state that can be written as a single ket $|\psi\rangle$. A **mixed state** arises from
classical uncertainty about which pure state the system is in.

The two situations look identical in many bases but differ in coherences: the off-diagonal entries
of the density matrix.

**Example:** the pure superposition $|{+}\rangle = \tfrac{1}{\sqrt{2}}(|0\rangle + |1\rangle)$ and
a 50/50 mixture of $|0\rangle$ and $|1\rangle$ both give equal Z-basis probabilities, yet they
behave completely differently in the X basis.

In [ ]:
# Pure state |+>
sv_plus = Statevector.from_label('+')
rho_pure = DensityMatrix(sv_plus)

print("Pure state |+>, density matrix:")
print(np.round(rho_pure.data, 4))
print()

In [ ]:
# Mixed state: 50/50 mixture of |0> and |1>
rho_0_arr = np.array([[1, 0], [0, 0]], dtype=complex)
rho_1_arr = np.array([[0, 0], [0, 1]], dtype=complex)
rho_mix = DensityMatrix(0.5 * rho_0_arr + 0.5 * rho_1_arr)

print("Mixed state (50/50 |0> and |1>), density matrix:")
print(np.round(rho_mix.data, 4))
print()
print("Both have diagonal [0.5, 0.5] — same Z-basis statistics.")
print("Only |+> has off-diagonals (coherences).")

In [ ]:
# Visualize: plot the real parts of three density matrices as heatmaps
fig, axes = plt.subplots(1, 3, figsize=(11, 3))

to_plot = [
    (rho_pure,  "|+⟩ (pure superposition)"),
    (rho_mix,   "50/50 mixture |0⟩, |1⟩"),
    (DensityMatrix(Statevector.from_label('0')), "|0⟩ (pure basis state)"),
]

for ax, (dm, title) in zip(axes, to_plot):
    im = ax.imshow(dm.data.real, vmin=-0.5, vmax=0.5, cmap='RdBu_r')
    ax.set_title(title, fontsize=9)
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(['|0⟩', '|1⟩'])
    ax.set_yticklabels(['⟨0|', '⟨1|'])
    plt.colorbar(im, ax=ax, shrink=0.8)

plt.suptitle("Real part of density matrix (red = positive, blue = negative)")
plt.tight_layout()
plt.show()

print("Observation: |+> has large off-diagonals (coherences).")
print("The mixture has none. |0> has all weight in the top-left entry.")

## 2. The Density Operator

For a pure state $|\psi\rangle$: $\rho = |\psi\rangle\langle\psi|$

For an ensemble $\{(p_i, |\psi_i\rangle)\}$: $\rho = \sum_i p_i |\psi_i\rangle\langle\psi_i|$

Every valid density operator is **Hermitian**, **positive semidefinite**, and has **unit trace**.

Key formulas:
- Expectation value: $\langle O\rangle = \mathrm{Tr}(O\rho)$
- Unitary evolution: $\rho \to U\rho U^\dagger$
- Measurement probability: $P(k) = \rho_{kk}$ (diagonal entry)
- Purity: $\gamma = \mathrm{Tr}(\rho^2) \in [1/d, 1]$

In [ ]:
# Verify the three properties of a density matrix
rho = rho_pure

is_hermitian = np.allclose(rho.data, rho.data.conj().T)
eigvals = np.linalg.eigvalsh(rho.data)
is_psd = np.all(eigvals >= -1e-10)
tr = rho.trace()

print(f"Hermitian:            {is_hermitian}")
print(f"Positive semidefinite: {is_psd}  (eigenvalues: {np.round(eigvals, 4)})")
print(f"Trace:                {tr.real:.6f}")
print(f"is_valid():           {rho.is_valid()}")

In [ ]:
# Purity of various states
states_to_check = [
    ("Pure |0>",            DensityMatrix(Statevector.from_label('0'))),
    ("Pure |+>",            DensityMatrix(Statevector.from_label('+'))),
    ("Mixed 50/50",         rho_mix),
    ("Maximally mixed I/2", DensityMatrix(np.eye(2) / 2)),
]

print(f"{'State':>25}  {'Purity':>8}  {'Pure?':>6}")
print("-" * 44)
for name, dm in states_to_check:
    p = dm.purity().real
    print(f"{name:>25}  {p:>8.4f}  {'yes' if abs(p - 1) < 1e-6 else 'no':>6}")

In [ ]:
# Demonstrate unitary evolution: apply H to |0> via rho.evolve()
rho_0 = DensityMatrix(Statevector.from_label('0'))

qc_h = QuantumCircuit(1)
qc_h.h(0)
rho_after_h = rho_0.evolve(qc_h)

print("rho_0 = |0><0|:")
print(np.round(rho_0.data, 4))
print()
print("After H gate (should equal |+><+|):")
print(np.round(rho_after_h.data, 4))

## 3. Partial Trace and Entanglement

The **partial trace** $\rho_A = \mathrm{Tr}_B(\rho_{AB})$ discards subsystem $B$ and returns
the best local description of $A$.

Key insight: an entangled pure state always yields **mixed** reduced states.
A qubit that has interacted with its environment appears mixed for exactly this reason:
its state is entangled with environmental degrees of freedom you cannot access.

In [ ]:
# Build the Bell state
qc_bell = QuantumCircuit(2)
qc_bell.h(0)
qc_bell.cx(0, 1)
rho_bell = DensityMatrix(qc_bell)

print("Bell state density matrix (rounded):")
print(np.round(rho_bell.data, 2))
print(f"\nGlobal purity: {rho_bell.purity().real:.4f}")

In [ ]:
# Partial trace: trace out qubit 1 (index 1), keep qubit 0
rho_A = partial_trace(rho_bell, [1])

print("Reduced state of qubit 0 (qubit 1 traced out):")
print(np.round(rho_A.data, 4))
print(f"Purity: {rho_A.purity().real:.4f}")
print(f"Von Neumann entropy: {entropy(rho_A, base=2):.4f} ebits")
print()
print("The global state is pure (purity=1).")
print("Each individual qubit is maximally mixed (purity=0.5, entropy=1).")

In [ ]:
# Compare entanglement entropy across different states

# Product state |+>|0>: no entanglement
qc_prod = QuantumCircuit(2)
qc_prod.h(0)
rho_prod = DensityMatrix(qc_prod)
rho_prod_A = partial_trace(rho_prod, [1])

# GHZ state (3 qubits)
qc_ghz = QuantumCircuit(3)
qc_ghz.h(0)
qc_ghz.cx(0, 1)
qc_ghz.cx(0, 2)
rho_ghz = DensityMatrix(qc_ghz)
rho_ghz_A = partial_trace(rho_ghz, [1, 2])   # trace out qubits 1 and 2

print(f"{'State':>22}  {'S(qubit 0) / ebits':>20}  {'Purity(qubit 0)':>16}")
print("-" * 62)
print(f"{'Product |+>|0>':>22}  "
      f"{entropy(rho_prod_A, base=2):>20.4f}  "
      f"{rho_prod_A.purity().real:>16.4f}")
print(f"{'Bell |Phi+>':>22}  "
      f"{entropy(rho_A, base=2):>20.4f}  "
      f"{rho_A.purity().real:>16.4f}")
print(f"{'GHZ (3 qubits)':>22}  "
      f"{entropy(rho_ghz_A, base=2):>20.4f}  "
      f"{rho_ghz_A.purity().real:>16.4f}")

In [ ]:
# Plot: purity of qubit 0 as entanglement with qubit 1 increases.
#
# Circuit: ry(2*theta) on q0, then CX(q0, q1).
# State:   cos(theta)|00> + sin(theta)|11>
# At theta=0:    product state |00>, qubit 0 is pure.
# At theta=pi/4: Bell state (|00>+|11>)/sqrt(2), qubit 0 is maximally mixed.
#
# Reduced state of q0: cos^2(theta)|0><0| + sin^2(theta)|1><1|
# Purity: cos^4(theta) + sin^4(theta)

thetas = np.linspace(0, np.pi / 4, 50)
purities = []

for theta in thetas:
    qc_interp = QuantumCircuit(2)
    qc_interp.ry(2 * theta, 0)   # creates cos(theta)|0> + sin(theta)|1> on q0
    qc_interp.cx(0, 1)           # entangles: cos(theta)|00> + sin(theta)|11>
    rho_interp = DensityMatrix(qc_interp)
    rho_A_interp = partial_trace(rho_interp, [1])
    purities.append(rho_A_interp.purity().real)

plt.figure(figsize=(7, 3))
plt.plot(np.degrees(thetas), purities, color='steelblue', linewidth=2)
plt.xlabel("Entangling angle θ (degrees)")
plt.ylabel("Purity of qubit 0")
plt.title("Purity decreases as entanglement with qubit 1 increases")
plt.axhline(0.5, color='gray', linestyle=':', linewidth=0.8, label='Maximally mixed (θ=45°)')
plt.legend()
plt.tight_layout()
plt.show()

print("At theta=0°:  product state, purity=1.0")
print("At theta=45°: Bell state,    purity=0.5  (maximally mixed)")

## 4. The DensityMatrix Class in Qiskit

`DensityMatrix` accepts a circuit, a `Statevector`, or a NumPy array.
It shares a common interface with `Statevector` and provides methods for
purity, measurement probabilities, and unitary evolution.

In [ ]:
# Three equivalent ways to build the Bell state density matrix

# 1. From a circuit
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
dm_from_circuit = DensityMatrix(qc)

# 2. From a Statevector
sv = Statevector([1/np.sqrt(2), 0, 0, 1/np.sqrt(2)])
dm_from_sv = DensityMatrix(sv)

# 3. From a raw NumPy array
mat = np.array([[0.5, 0, 0, 0.5],
                [0,   0, 0, 0  ],
                [0,   0, 0, 0  ],
                [0.5, 0, 0, 0.5]], dtype=complex)
dm_from_array = DensityMatrix(mat)

print("All three are equal:", np.allclose(dm_from_circuit.data, dm_from_sv.data))
print("Valid density matrix:", dm_from_circuit.is_valid())

In [ ]:
# Key methods
dm = DensityMatrix(qc)   # Bell state

print(f"purity():     {dm.purity().real:.4f}")
print(f"trace():      {dm.trace().real:.4f}")
print(f"is_valid():   {dm.is_valid()}")
print()
print("probabilities_dict():")
for bitstr, p in sorted(dm.probabilities_dict().items()):
    print(f"  |{bitstr}> : {p:.4f}")

In [ ]:
# Evolve the Bell state by applying X to qubit 0: |Phi+> -> |Psi+>
qc_x0 = QuantumCircuit(2)
qc_x0.x(0)
rho_psi_plus = rho_bell.evolve(qc_x0)

print("Bell state |Phi+> after X on qubit 0:")
for bitstr, p in sorted(rho_psi_plus.probabilities_dict().items()):
    print(f"  |{bitstr}> : {p:.4f}")
print("(|Phi+> = (|00>+|11>)/sqrt(2) has become |Psi+> = (|10>+|01>)/sqrt(2))")

## 5. Fidelity and Trace Distance

**Fidelity** $F(\rho, \sigma) \in [0,1]$: measures overlap. $F=1$ iff $\rho = \sigma$.

For pure states: $F(|\psi\rangle, |\phi\rangle) = |\langle\psi|\phi\rangle|^2$

For mixed states (Uhlmann): $F(\rho, \sigma) = \left(\mathrm{Tr}\sqrt{\sqrt{\rho}\,\sigma\,\sqrt{\rho}}\right)^2$

---

**Trace distance** $D(\rho, \sigma) = \tfrac{1}{2}\mathrm{Tr}|\rho - \sigma| \in [0,1]$:
the maximum probability advantage any measurement has in distinguishing the two states.

**Fuchs-van de Graaf inequalities:**
$1 - \sqrt{F} \leq D \leq \sqrt{1 - F}$

For pure-state pairs the upper bound is tight: $D = \sqrt{1-F}$.

In [ ]:
rho_0    = DensityMatrix(Statevector.from_label('0'))
rho_1    = DensityMatrix(Statevector.from_label('1'))
rho_p    = DensityMatrix(Statevector.from_label('+'))
rho_maxmix = DensityMatrix(np.eye(2) / 2)

print("Fidelity examples:")
print(f"  F(|0>, |0>)  = {state_fidelity(rho_0, rho_0):.4f}  (identical)")
print(f"  F(|0>, |1>)  = {state_fidelity(rho_0, rho_1):.4f}  (orthogonal)")
print(f"  F(|0>, |+>)  = {state_fidelity(rho_0, rho_p):.4f}  (|<0|+>|^2 = 1/2)")
print(f"  F(|0>, I/2)  = {state_fidelity(rho_0, rho_maxmix):.4f}  (pure vs maximally mixed)")
print(f"  F(|+>, I/2)  = {state_fidelity(rho_p, rho_maxmix):.4f}  (pure vs maximally mixed)")

In [ ]:
def trace_distance(rho, sigma):
    """D(rho, sigma) = (1/2) * sum |eigenvalues(rho - sigma)|."""
    diff = rho.data - sigma.data
    eigvals = np.linalg.eigvalsh(diff)
    return 0.5 * float(np.sum(np.abs(eigvals)))


print("Trace distance examples:")
print(f"  D(|0>, |0>)  = {trace_distance(rho_0, rho_0):.4f}  (identical)")
print(f"  D(|0>, |1>)  = {trace_distance(rho_0, rho_1):.4f}  (perfectly distinguishable)")
print(f"  D(|0>, |+>)  = {trace_distance(rho_0, rho_p):.4f}")
print(f"  D(|+>, I/2)  = {trace_distance(rho_p, rho_maxmix):.4f}")

In [ ]:
# Verify Fuchs-van de Graaf inequalities for |0> vs |+>
f = state_fidelity(rho_0, rho_p)
d = trace_distance(rho_0, rho_p)

print(f"Fuchs-van de Graaf for |0> vs |+>:")
print(f"  lower = 1 - sqrt(F) = {1 - np.sqrt(f):.4f}")
print(f"  D                   = {d:.4f}")
print(f"  upper = sqrt(1 - F) = {np.sqrt(1 - f):.4f}")
print(f"  Upper bound tight (pure pair): {np.isclose(d, np.sqrt(1 - f))}")

In [ ]:
# Plot fidelity and trace distance vs rotation angle theta
# State: cos(theta)|0> + sin(theta)|1>, compared with |0>

thetas = np.linspace(0, np.pi, 100)
fidelities = []
distances  = []

for theta in thetas:
    sv_rot = Statevector([np.cos(theta), np.sin(theta)])
    rho_rot = DensityMatrix(sv_rot)
    f_val = state_fidelity(rho_0, rho_rot)
    d_val = trace_distance(rho_0, rho_rot)
    fidelities.append(f_val)
    distances.append(d_val)

plt.figure(figsize=(8, 3.5))
plt.plot(np.degrees(thetas), fidelities, label='Fidelity F', color='steelblue', linewidth=2)
plt.plot(np.degrees(thetas), distances,  label='Trace distance D', color='tomato', linewidth=2)
plt.xlabel("Rotation angle θ (degrees)")
plt.ylabel("Value")
plt.title("F and D between |0⟩ and cos(θ)|0⟩ + sin(θ)|1⟩")
plt.legend()
plt.tight_layout()
plt.show()

print("At theta=0: identical states (F=1, D=0).")
print("At theta=pi/2: |0> vs |1> — orthogonal (F=0, D=1).")
print("D = sqrt(1-F) holds everywhere since both states are pure.")

## 6. Exercises

Work through each exercise in the cell below it. Starter code is provided.
Expected numerical results are given at the end so you can check your answers.

### Exercise 1: Purity of a depolarized state

The **depolarizing channel** with parameter $p$ maps $|0\rangle\langle 0|$ to:
$$\rho(p) = (1 - p)\,|0\rangle\langle 0| + \frac{p}{2}\,I$$

For $p=0$ the state is pure; for $p=1$ it is maximally mixed.

1. Implement `depolarized(p)` that returns the density matrix above.
2. Plot purity as a function of $p \in [0, 1]$.
3. At what value of $p$ does the purity first drop below 0.6?

*Expected answer for (3): p ≈ 0.553*

In [ ]:
# Exercise 1 — Purity of a depolarized state

def depolarized(p):
    """Return DensityMatrix for (1-p)|0><0| + (p/2)*I."""
    # TODO: implement this
    pass


# TODO: compute purity for p in [0, 1] and plot
p_values = np.linspace(0, 1, 200)
# purities = [depolarized(p).purity().real for p in p_values]

# TODO: find p where purity first drops below 0.6
# threshold = 0.6
# p_threshold = ...

### Exercise 2: Partial trace of the W state

The **W state** is $|W\rangle = \tfrac{1}{\sqrt{3}}(|001\rangle + |010\rangle + |100\rangle)$.

In Qiskit's ordering (qubit 0 = LSB = rightmost bit in ket label), the W state statevector
has nonzero amplitudes at indices 1 (|001⟩), 2 (|010⟩), and 4 (|100⟩).

1. Build the W state as a `DensityMatrix` from its statevector.
2. Compute the reduced density matrix and von Neumann entropy of qubit 0.
3. Compare with the Bell state result. Which system is more entangled per qubit?

*Expected entropy of qubit 0 in W state: ≈ 0.918 ebits*

In [ ]:
# Exercise 2 — W state partial trace

# The W state: |W> = (|001> + |010> + |100>) / sqrt(3)
# In Qiskit index ordering: indices 1, 2, 4 are nonzero
w_vec = np.zeros(8, dtype=complex)
# TODO: fill in the nonzero amplitudes at indices 1, 2, 4
# w_vec[1] = ...
# w_vec[2] = ...
# w_vec[4] = ...

# TODO: create DensityMatrix, compute partial trace and entropy
# rho_W = DensityMatrix(Statevector(w_vec))
# rho_W_qubit0 = partial_trace(rho_W, [1, 2])  # trace out qubits 1 and 2
# S_W = entropy(rho_W_qubit0, base=2)
# print(f"W state entropy of qubit 0: {S_W:.4f} ebits")

### Exercise 3: Fidelity and trace distance for a mixed state pair

Consider two single-qubit states:
$$\rho_1 = 0.8\,|0\rangle\langle 0| + 0.2\,|1\rangle\langle 1|$$
$$\rho_2 = 0.6\,|0\rangle\langle 0| + 0.4\,|1\rangle\langle 1|$$

1. Compute $F(\rho_1, \rho_2)$ and $D(\rho_1, \rho_2)$.
2. Verify that the Fuchs-van de Graaf inequalities hold.
3. Are the inequalities tight? Why or why not?

*Expected: F ≈ 0.9520, D = 0.2*

In [ ]:
# Exercise 3 — Fidelity and trace distance for a mixed pair

# TODO: build rho_1 and rho_2 as DensityMatrix objects
# rho_1 = DensityMatrix(...)   # 0.8|0><0| + 0.2|1><1|
# rho_2 = DensityMatrix(...)   # 0.6|0><0| + 0.4|1><1|

# TODO: compute fidelity and trace distance
# f12 = state_fidelity(rho_1, rho_2)
# d12 = trace_distance(rho_1, rho_2)

# TODO: check Fuchs-van de Graaf
# lower = 1 - np.sqrt(f12)
# upper = np.sqrt(1 - f12)
# print(f"lower={lower:.4f} <= D={d12:.4f} <= upper={upper:.4f}")

## Summary

In this lesson you:

- Built density matrices for pure states (outer products $|\psi\rangle\langle\psi|$) and mixed states
  (weighted sums of projectors), and identified coherences as the physically distinguishing feature.
- Verified the three defining properties of density operators: Hermitian, positive semidefinite,
  unit trace. Computed purity $\mathrm{Tr}(\rho^2)$ and confirmed it equals 1 only for pure states.
- Applied the partial trace to the Bell state and observed that a globally pure entangled state
  yields maximally mixed reduced states. Computed von Neumann entropy as an entanglement measure.
- Used Qiskit's `DensityMatrix`, `partial_trace`, `entropy`, and `state_fidelity` functions,
  verifying that circuits, statevectors, and NumPy arrays all produce the same object.
- Computed fidelity and trace distance for several state pairs and verified the Fuchs-van de Graaf
  inequalities, confirming the tight relationship $D = \sqrt{1-F}$ for pure-state pairs.

**Lesson 2** introduces quantum channels: the Kraus operator representation of noise,
the depolarizing, bit-flip, and amplitude-damping channels, and custom noise models in `AerSimulator`.